# 56 — Phase 1 Bundle C: Chat-history windowing on Blind-A

**The cheapest possible single-axis test of a Phase 0 finding.** Single change vs the May 15 v5-kto submission (composite 0.21):
- New config field: `chat_history_window: 3` → use last 3 turn-pairs (6 messages) instead of full history

Nothing else changes. Same retriever (wrrf_bm25_dense_lyrics_v1), same CMQR, same ProRank, same v5-kto-merged 3B responder.

**Why we expect this to help**: Phase 0 diagnostic on 8000 dev turns showed:
- Long queries (>30 words): recall@20 = **0.247**
- Medium queries (10–30 words): recall@20 = 0.281
- Short queries (<10 words): recall@20 = **0.359**

An 11pp recall@20 spread driven by chat-history length. Conversation history gets concatenated into the retrieval input; long histories add off-topic noise that pulls retrieval away from current intent. Windowing to the last 3 turn-pairs is the standard conversational-IR fix.

**Compute cost: zero new** — no catalog re-embed, no query generation, no new model download. Just one config knob.

**Wallclock estimate** (same as notebook 53's config 132, since the model + retrievers are identical):
| GPU | Total (clone + auth + deps + inference + package) |
|---|---|
| Blackwell-95GB / A100 | ~40–60 min |
| L4 | ~1–2 hr |

**Decision rule**:
| Outcome | Action |
|---|---|
| Composite ≥ 0.23 (≥+0.02 over v5-kto 0.21) | Windowing helps — ship; combine with BGE-M3 or doc2query next |
| Composite in 0.21 ± 0.05 | Roughly neutral; cheap so keep it on, move to other levers |
| Composite < 0.16 | Windowing hurts — revert; responder needs full history more than retrieval benefits |

In [ ]:
# 1) GPU check.
!nvidia-smi | head -10

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

In [ ]:
# 3) HF auth + Drive mount (for HF model cache + submissions backup).
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab secrets.')
except Exception as e:
    print('NO HF_TOKEN — set it in Colab secrets.', e)

from google.colab import drive
drive.mount('/content/drive')
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
print('HF_HOME =', os.environ['HF_HOME'])

In [ ]:
# 4) Install deps. No vLLM, no sentence-transformers needed (no BGE-M3 / doc2query here).
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf pyyaml bm25s scipy numpy

In [ ]:
# 5) Run config 150 inference on Blind-A.
# Only change vs the May 15 v5-kto submission: chat_history_window: 3 in the YAML.
%cd /content/recsys2026/music-crs-baselines
TID = '150-v5kto-windowed-prorank-rerank-blindsetA'
PRED = f'exp/inference/blindset_A/{TID}.json'
import os
if os.path.exists(PRED):
    print(f'Predictions exist at {PRED}. To force re-run: !rm', PRED)
else:
    print(f'Running config {TID} on Blind-A (~45-90 min on L4, ~30-45 min on Blackwell)...')
    !python run_inference_blindset.py --tid {TID} --batch_size 16
%cd /content/recsys2026

In [ ]:
# 6) Validate schema + package CodaBench-ready zip.
from datetime import date
import sys, os, shutil
sys.path.insert(0, '/content/recsys2026/scripts')
from validate_prediction import load_prediction, validate_schema, package_zip

PRED_ABS = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/150-v5kto-windowed-prorank-rerank-blindsetA.json'
ZIP_PATH = f'/content/recsys2026/data/submissions/blindset_A_{date.today().isoformat()}_150_v5kto_windowed.zip'
os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)

predictions = load_prediction(PRED_ABS)
errors = validate_schema(predictions, 'blindA')
if errors:
    print('SCHEMA FAILED:')
    for e in errors[:20]: print(f'  - {e}')
    raise SystemExit('Refusing to package invalid predictions.')

package_zip(PRED_ABS, ZIP_PATH)
print(f'\nReady for CodaBench: {ZIP_PATH}')
print(f'  size: {os.path.getsize(ZIP_PATH):,} bytes, {len(predictions)} predictions')

DRIVE_SUBS = '/content/drive/MyDrive/recsys2026_submissions'
os.makedirs(DRIVE_SUBS, exist_ok=True)
drive_copy = os.path.join(DRIVE_SUBS, os.path.basename(ZIP_PATH))
shutil.copy(ZIP_PATH, drive_copy)
print(f'  also at: {drive_copy}')

## Submit
1. Download zip from `/content/drive/MyDrive/recsys2026_submissions/`
2. Upload to CodaBench → wait ~5 min
3. Compare composite to your prior v5-kto 0.21 (this submission's only change vs that is `chat_history_window: 3`)

**Next moves depending on result**:
- **Helps (≥+0.02)**: combine with BGE-M3 (build config 152 = bge_m3 + v5kto + windowed) or doc2query when ready
- **Neutral**: keep windowing on by default (no downside; small or no upside), move to CLAP audio integration (B1) which adds a fundamentally different signal
- **Hurts**: revert — responder needed the full history more than retrieval benefited from windowing. Move to CLAP audio.